## <span style="color:#50C878; font-weight:bold;">Load in Data</span>

In [242]:
%pip install matplotlib
%pip install seaborn


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: matplotlib in c:\users\pcn\appdata\local\programs\python\python311\lib\site-packages (3.10.0)




[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [243]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import RFE
import numpy as np
from sklearn.feature_selection import VarianceThreshold
import joblib
from collections import defaultdict
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Load the dataset
data = pd.read_csv('Car_Kh24.csv')

df = pd.DataFrame(data)

df.info()
# data.isnull().sum()
# df.nunique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17873 entries, 0 to 17872
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Ad ID         17873 non-null  int64  
 1   Category      17873 non-null  object 
 2   Locations     17873 non-null  object 
 3   Car Makes     17870 non-null  object 
 4   Car Model     17750 non-null  object 
 5   Year          17871 non-null  float64
 6   Tax Type      17873 non-null  object 
 7   Condition     17873 non-null  object 
 8   Body Type     14215 non-null  object 
 9   Fuel          15712 non-null  object 
 10  Transmission  16387 non-null  object 
 11  Color         17685 non-null  object 
 12  Link          17873 non-null  object 
 13  Title         17873 non-null  object 
 14  Price         17873 non-null  object 
 15  Year Used     17871 non-null  float64
dtypes: float64(2), int64(1), object(13)
memory usage: 2.2+ MB


In [244]:
# unique_category = df.Category.unique()
# print(unique_category)


## <span style="color:#50C878; font-weight:bold;">Data Preprocessing</span>

### <span style="color:#50C878; font-weight:bold;">Remove NULL Row</span>


In [245]:
feature_has_missing_values = [
    feature for feature in df.columns if df[feature].isnull().any()
]
df = df.dropna(subset=feature_has_missing_values, how="any")
df.isnull().sum()

Ad ID           0
Category        0
Locations       0
Car Makes       0
Car Model       0
Year            0
Tax Type        0
Condition       0
Body Type       0
Fuel            0
Transmission    0
Color           0
Link            0
Title           0
Price           0
Year Used       0
dtype: int64

### <span style="color:#50C878; font-weight:bold;">Remove Duplicate Data</span>


In [246]:
# find duplicates
duplicates = df[df.duplicated()]

# count duplicates
print(f"Number of duplicate rows: {len(duplicates)}")

#drop duplicates
df = df.drop_duplicates()

Number of duplicate rows: 3


### <span style="color:#50C878; font-weight:bold;">Remove Unknown Data</span>


In [247]:
def check_column_value(df, value):
    """
    Checks if each column in a pandas DataFrame contains a specific value and prints the column names that contain the value.
    
    Parameters:
        df (pandas DataFrame): The DataFrame to check.
        value (any type): The value to look for in each column.
    """
    values = value if isinstance(value, list) else [value]
    matching_columns = df.apply(lambda col: col.isin(values).any())
    matching_columns = matching_columns[matching_columns]

    count = defaultdict(list)
    index = set()

    for val in values:
        for col in matching_columns.index:
            count[col].append(df[df[col] == val].shape[0])
            index.update(df[df[col] == val].index)

    print(f"\033[94mColumns that contain {values}:\n")
    print(pd.Series(count))

    return index

In [248]:
index = check_column_value(df, ['Other', 'Other - ផ្សេងៗ'])
df = df.drop(index, axis='rows')

Columns that contain ['Other', 'Other - ផ្សេងៗ']:

Car Model     [0, 53]
Body Type    [920, 0]
Color        [274, 0]
dtype: object


### <span style="color:#50C878; font-weight:bold;">Drop Unused Columns</span>

In [249]:
df.columns = df.columns.str.strip()

# columns_to_drop = ['Ad ID', 'Category', 'Locations', 'Fuel', 'Transmission', 'Color', 'Link', 'Title', 'Year Used']
columns_to_drop = ['Ad ID', 'Category', 'Locations', 'Link', 'Title']

df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

df.columns

Index(['Car Makes', 'Car Model', 'Year', 'Tax Type', 'Condition', 'Body Type',
       'Fuel', 'Transmission', 'Color', 'Price', 'Year Used'],
      dtype='object')

### <span style="color:#50C878; font-weight:bold;">Convert Price from String to Float</span>

In [250]:
df['Price'] = df['Price'].apply(lambda x: x.split('$')[1].replace(',', '')).astype(float)
df['Price'].dtype

dtype('float64')

### <span style="color:#50C878; font-weight:bold;">Convert Year from Float to Integer</span>

In [251]:
# df["Year"] = df["Year"].astype(int)
# df.Year.dtype

### <span style="color:#50C878; font-weight:bold;">Feature Engineering</span>

In [252]:
# Add 'Car Age' feature
df['Car Age'] = 2025 - df['Year']

# Add 'Price per Year' feature
# df['Price per Year'] = df['Price'] / (2025 - df['Year'])

### <span style="color:#50C878; font-weight:bold;">Binning</span>

In [253]:
# z = round(df.groupby(['Car Makes'])['Price'].agg(['mean']), 2)
# min, max = z.agg(['min','max']).values
# df = df.merge(z, how='left', on='Car Makes')

# bin_width = (min + max) / 3
# bins = [i for i in range(int(24600.9100), int(np.ceil(max)), int(np.ceil(bin_width)))]
# bins[-1] = int(np.ceil(max))
# bins.insert(0, 0)

# cars_bin = ['Budget', 'Medium', 'High-End']
# df['Price Range'] = pd.cut(df['mean'], bins=bins, right=False, labels=cars_bin)
# # df.head()

### <span style="color:#50C878; font-weight:bold;">Handle Skewed Data</span>

In [254]:
def boxcox(new_df, col):

    # Box-Cox transformation
    transformed_data, lambda_param = stats.boxcox(new_df[col])

    Boxcox = pd.Series(transformed_data)
    
    t=sns.distplot(Boxcox,label="Skewness: %.2f"%(Boxcox.skew()) )
    t.legend()
    return Boxcox, lambda_param

price_boxcox, price_lambda = boxcox(df, 'Price')
print(price_lambda)

C:\Users\PCN\AppData\Local\Temp\ipykernel_12992\2260968436.py:8: UserWarning: 

`distplot` is a deprecated function and will be removed in seaborn v0.14.0.

Please adapt your code to use either `displot` (a figure-level function with
similar flexibility) or `histplot` (an axes-level function for histograms).

For a guide to updating your code to use the new functions, please see
https://gist.github.com/mwaskom/de44147ed2974457ad6372750bbe5751

  t=sns.distplot(Boxcox,label="Skewness: %.2f"%(Boxcox.skew()) )


<IPython.core.display.Javascript object>

-0.02832931484315432


In [255]:
year_boxcox, year_lambda = boxcox(df, 'Year')

C:\Users\PCN\AppData\Local\Temp\ipykernel_12992\2260968436.py:8: UserWarning: 

`distplot` is a deprecated function and will be removed in seaborn v0.14.0.

Please adapt your code to use either `displot` (a figure-level function with
similar flexibility) or `histplot` (an axes-level function for histograms).

For a guide to updating your code to use the new functions, please see
https://gist.github.com/mwaskom/de44147ed2974457ad6372750bbe5751

  t=sns.distplot(Boxcox,label="Skewness: %.2f"%(Boxcox.skew()) )


<IPython.core.display.Javascript object>

In [256]:
df_boxcox = df.copy()
df_boxcox['Price'] = list(price_boxcox)
df_boxcox.shape, df.shape

((11859, 12), (11859, 12))

### <span style="color:#50C878; font-weight:bold;">Detect Outlier</span>

In [257]:
# func to Inter-quartile Range Method
def out_iqr(df , column):
    global lower,upper
    q25, q75 = np.quantile(df[column], 0.25), np.quantile(df[column], 0.75)
    # calculate the IQR
    iqr = q75 - q25
    # calculate the outlier cutoff
    cut_off = iqr * 1.5
    # calculate the lower and upper bound value
    lower, upper = q25 - cut_off, q75 + cut_off
    print('The IQR is',iqr)
    print('The lower bound value is', lower)
    print('The upper bound value is', upper)
    # Calculate the number of records below and above lower and above bound value respectively
    df1 = df[df[column] > upper]
    df2 = df[df[column] < lower]
    return print('Total number of outliers are', df1.shape[0]+ df2.shape[0])

In [258]:
def visualize_outlier(df, col):
    out_iqr(df, col)
    plt.figure(figsize = (10,6))
    sns.distplot(df[col], kde=False)
    plt.axvspan(xmin = lower,xmax= df.Price.min(),alpha=0.2, color='red')
    plt.axvspan(xmin = upper,xmax= df.Price.max(),alpha=0.2, color='red')
    plt.show()

In [259]:
# visualize_outlier(df_boxcox, 'Price')

In [260]:
# visualize_outlier(df, 'Price')

### <span style="color:#50C878; font-weight:bold;">Remove Outlier</span>

In [261]:
def outlier_detection(df):
    Q1 = df.quantile(0.25)
    Q3 = df.quantile(0.75)
    IQR = Q3 - Q1
    upper_end = Q3 + 1.5 * IQR
    lower_end = Q1 - 1.5 * IQR
    outlier = df[(df > upper_end) | (df < lower_end)]
    return outlier

In [262]:
# # Define price threshold (optional, based on your data)
# price_lower_threshold = df['Price'].quantile(0.25)
# price_upper_threshold = df['Price'].quantile(0.75)

# # Filter out rows where price is out of the defined range
# df = df[(df['Price'] > price_lower_threshold) & (df['Price'] < price_upper_threshold)]

outlier_index = list(outlier_detection(df_boxcox['Price']).index)
df = df_boxcox.drop(outlier_index, axis= 'rows')

### <span style="color:#50C878; font-weight:bold;">Encoding</span>

In [263]:
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
categorical_features

['Car Makes',
 'Car Model',
 'Tax Type',
 'Condition',
 'Body Type',
 'Fuel',
 'Transmission',
 'Color']

In [264]:
# # Label Encoding
# le = LabelEncoder()

# # Columns to encode
# categorical_cols = ['Car Model']

# # Apply label encoding to categorical columns
# for col in categorical_cols:
#     df[col] = le.fit_transform(df[col])

# # One-Hot Encoding
# df = pd.get_dummies(df, columns=['Car Makes', 'Tax Type', 'Condition', 'Body Type'])

df = pd.get_dummies(df, columns=categorical_features)

# # df.head()

# Target Encoding
# target_encoder = ce.TargetEncoder(cols= categorical_features)

## <span style="color:#50C878; font-weight:bold;">Model</span>

### <span style="color:#50C878; font-weight:bold;">Train the Elastic Net Regression Model</span>

In [265]:
X = df.drop(columns=['Price'])
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler for future use
# joblib.dump(scaler, 'scaler.pkl')
# print("Scaler saved as 'scaler.pkl'")

In [266]:
# Save the entire X_train DataFrame to a CSV file
# X_train.to_csv('X_train_data.csv', index=False)
# 

In [267]:
%pip install tensorflow


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [268]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Initialize the model
model = Sequential()

# Add input layer (first hidden layer)
model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))

# Add hidden layers
# model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
# model.add(Dense(4, activation='relu'))
# model.add(Dense(2, activation='relu'))

# Add output layer (predicting price)
model.add(Dense(1))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# Summary of the model
model.summary()


c:\Users\PCN\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_35 (Dense)                │ (None, 64)             │        33,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,337 (157.57 KB)

 Trainable params: 40,337 (157.57 KB)

 Non-trainable params: 0 (0.00 B)

In [269]:
# from sklearn.metrics import r2_score

# # Compile the model with MAE as a metric for regression tasks
# model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
# # 
# # Train the model and store history
# history = model.fit(X_train, y_train, epochs=200, batch_size=32, validation_data=(X_test, y_test), verbose=1)
# joblib.dump(model, 'nural_network1.pkl')


In [270]:
model = joblib.load('nural_network1.pkl')

In [271]:
print(X_test.columns)



Index(['Year', 'Year Used', 'Car Age', 'Car Makes_Acura',
       'Car Makes_Alfa Romeo', 'Car Makes_Aston Martin', 'Car Makes_Audi',
       'Car Makes_BAIC', 'Car Makes_BESTUNE', 'Car Makes_BMW',
       ...
       'Color_Brown', 'Color_Gold', 'Color_Gray', 'Color_Green',
       'Color_Orange', 'Color_Purple', 'Color_Red', 'Color_Silver',
       'Color_White', 'Color_Yellow'],
      dtype='object', length=517)


In [272]:
# Test
# Predict car prices
predictions = model.predict(X_test)

# Calculate R-squared (R² score)
r2 = r2_score(y_test, predictions.flatten())
# print(f"Mean Squared Error: {history.history['loss'][-1]}")
print(f"R-squared (R²) Score: {r2}")

# Show the predictions alongside the actual values
results = pd.DataFrame({'Actual': y_test, 'Predicted': predictions.flatten()})
print(len(results))
print(results.head(10))

73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
R-squared (R²) Score: 0.9136837848367699
2327
         Actual  Predicted
272    9.265287   9.262579
14650  9.446470   9.284052
8748   8.690102   8.289339
17476  8.584714   8.501305
12388  9.094182   9.147760
6706   9.207493   9.274832
12343  8.083305   8.163716
8954   8.932544   8.869559
46     8.534474   8.674518
8176   8.508009   8.573083


In [273]:
# Inverse Box-Cox transformation for 'Price'
def inverse_boxcox(transformed_data, lambda_param):
    if lambda_param == 0:
        print("hello")
        return np.exp(transformed_data)
        
    else:
        print("hi")
        return (lambda_param * transformed_data + 1) ** (1 / lambda_param)

# Inverse transformation for actual y_test (price) values
y_test_original_price = inverse_boxcox(y_test, price_lambda)

# Inverse transformation for predictions
print(predictions.flatten())
predictions_original_price = inverse_boxcox(predictions.flatten(), price_lambda)

# Show the predictions alongside the actual values in the original price scale
results_original_price = pd.DataFrame({'Actual': y_test_original_price, 'Predicted': predictions_original_price})

# Display the last 10 predictions
print("Predictions in original price scale:")
print(results_original_price.head(10))

# Calculate Mean Squared Error and R-squared on the original price scale
mse_original = mean_squared_error(y_test_original_price, predictions_original_price)
r2_original = r2_score(y_test_original_price, predictions_original_price)

print(f"Mean Squared Error (Original Price Scale): {mse_original}")
print(f"R-squared (R²) Score (Original Price Scale): {r2_original}")


hi
[9.262579  9.284052  8.289339  ... 7.1462536 8.63225   8.820278 ]
hi
Predictions in original price scale:
        Actual     Predicted
272    46500.0  46329.589886
14650  59500.0  47698.733638
8748   21500.0  12684.374940
17476  18700.0  16751.397592
12388  36900.0  39664.534548
6706   42999.0  47105.818951
12343   9700.0  10767.959278
8954   29700.0  27301.040023
46     17500.0  21060.202932
8176   16900.0  18414.855340
Mean Squared Error (Original Price Scale): 61218445.55525024
R-squared (R²) Score (Original Price Scale): 0.9159482702948883


In [274]:
import pandas as pd

# Example real data (from your input)
real_data = {
    # 'Price': ['$192,000'],  # Price in string format
    'Year': 2010,
    'Car Makes': 'Toyota',
    'Tax Type': 'Tax Paper',
    'Condition': 'New',
    'Body Type': 'SUV',
    'Fuel': 'Petrol',
    'Transmission': 'Auto',
    'Color': 'Color_White',
    'Model': 'Prius',
}

# Convert to DataFrame
real_df = pd.DataFrame([real_data])

# Convert 'Price' to numeric (removing '$' and commas)
# real_df['Price'] = real_df['Price'].apply(lambda x: x.split('$')[1].replace(',', '')).astype(float)

# Convert 'Year' to integer (if necessary)
real_df['Year'] = real_df['Year'].astype(int)

# Add 'Car Age' and 'Price per Year' feature (same as during preprocessing)
real_df['Car Age'] = 2025 - real_df['Year']

# Apply one-hot encoding (this should match the encoding done during training)
real_df = pd.get_dummies(real_df, columns=['Car Makes', 'Tax Type', 'Condition', 'Body Type', 'Fuel', 'Transmission', 'Color', 'Model'])

# Load the columns from the training data (from your model's feature set)
# Here we assume that the model expects columns that match the training data features.
train_columns = df.columns
real_df = real_df.replace({1: True, 0: False})
# Ensure real_df has all the columns from the training data and fill missing columns with zeros
real_df = real_df.reindex(columns=train_columns, fill_value=0)

# Show the transformed real data with the correct column alignment
# print(real_df.head())

# Convert real_df to a NumPy array
real_df_array = real_df.to_numpy()




In [275]:
print(X_train.columns)
import pandas as pd

# Assuming X_train.columns contains the column names
columns = list(X_train.columns)

# Convert the column names into a DataFrame for saving as CSV
columns_df = pd.DataFrame(columns, columns=["Column Names"])

# Save to CSV
columns_df.to_csv("./data/columns_list.csv", index=False)

print("Column names saved to './data/columns_list.csv'")


Index(['Year', 'Year Used', 'Car Age', 'Car Makes_Acura',
       'Car Makes_Alfa Romeo', 'Car Makes_Aston Martin', 'Car Makes_Audi',
       'Car Makes_BAIC', 'Car Makes_BESTUNE', 'Car Makes_BMW',
       ...
       'Color_Brown', 'Color_Gold', 'Color_Gray', 'Color_Green',
       'Color_Orange', 'Color_Purple', 'Color_Red', 'Color_Silver',
       'Color_White', 'Color_Yellow'],
      dtype='object', length=517)
Column names saved to './data/columns_list.csv'


In [276]:
import pandas as pd

# Example real data (from your input)
real_data = {
    'Year': 2010,
    'Car Makes': 'Toyota',
    'Tax Type': 'Tax Paper',
    'Condition': 'New',
    'Body Type': 'SUV',
    'Fuel': 'Petrol',
    'Transmission': 'Auto',
    'Color': 'Color_White',
    'Model': 'Prius',
}

real_df = pd.DataFrame([real_data])

# Apply one-hot encoding for categorical columns
real_df = pd.get_dummies(real_df, columns=['Car Makes', 'Tax Type', 'Condition', 'Body Type', 'Fuel', 'Transmission', 'Color', 'Model'])

# Feature engineering (Add 'Car Age' feature)
real_df['Car Age'] = float(2025 - real_df['Year'])
real_df['Year'] = float(real_df['Year'])

# Assuming 'df' is the training dataset that contains the column names
# If 'df' is not defined here, use your actual training dataset (e.g., df_train) that was used for training
train_columns = df.columns  # Replace df with the training dataset that has the columns

# Ensure real_df has all the columns from the training data and fill missing columns with zeros
real_df = real_df.reindex(columns=train_columns, fill_value=0)

# Drop 'Price' column, as it's the target variable and should not be part of the feature set
real_df = real_df.drop(columns=['Price'], errors='ignore')

# Convert real_df to a NumPy array for prediction
real_df_array = real_df.to_numpy()

# Output the result for verification
print(real_df_array)


[[2010.0 0 15.0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 True 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

C:\Users\PCN\AppData\Local\Temp\ipykernel_12992\652119679.py:22: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  real_df['Car Age'] = float(2025 - real_df['Year'])
C:\Users\PCN\AppData\Local\Temp\ipykernel_12992\652119679.py:23: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  real_df['Year'] = float(real_df['Year'])


In [277]:
# Print the shape to check if it's still empty
print("Shape of real_df after reindexing:", real_df.shape)

# Proceed if there are rows in real_df
if real_df.shape[0] > 0:
    # Scale the transformed data using the scaler
    real_df_scaled = scaler.transform(real_df_array)
    predicted_price = model.predict(real_df)
    print(predicted_price)

    # Assuming you have the lambda used for the Box-Cox transformation during training

    # Convert the predicted price back to the original scale
    predicted_price_original = inverse_boxcox(predicted_price.flatten(), price_lambda)

    # Output the predicted price
    print(f"The predicted price for the car is: ${predicted_price_original[0]:,.2f}")
else:
    print("The DataFrame is empty, please check your data.")

Shape of real_df after reindexing: (1, 517)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
[[9.041712]]
hi
The predicted price for the car is: $34,384.38


c:\Users\PCN\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [278]:
%pip install tk
%pip install PySimpleGUI
%pip install PyQt5

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [279]:
# First cell - run this separately
%matplotlib notebook
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox

In [280]:
# import tkinter as tk
# from tkinter import ttk
# from tkinter import messagebox

# def create_form():
#     root = tk.Tk()
#     root.title("Car Details Form")
#     root.geometry("400x600")

#     # Create main frame
#     main_frame = ttk.Frame(root, padding="10")
#     main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))

#     # Variables for form fields
#     year_var = tk.StringVar(value="2010")
#     make_var = tk.StringVar(value="Toyota")
#     tax_var = tk.StringVar(value="Tax Paper")
#     condition_var = tk.StringVar(value="New")
#     body_var = tk.StringVar(value="SUV")
#     fuel_var = tk.StringVar(value="Petrol")
#     transmission_var = tk.StringVar(value="Auto")
#     color_var = tk.StringVar(value="White")
#     model_var = tk.StringVar(value="Prius")

#     # Lists for dropdowns
#     years = list(range(2000, 2026))
#     car_makes = ['Toyota', 'Honda', 'Ford', 'BMW', 'Mercedes']
#     tax_types = ['Tax Paper', 'No Paper']
#     conditions = ['New', 'Used']
#     body_types = ['SUV', 'Sedan', 'Hatchback', 'Pickup']
#     fuels = ['Petrol', 'Diesel', 'Hybrid', 'Electric']
#     transmissions = ['Auto', 'Manual']
#     colors = ['White', 'Black', 'Silver', 'Red', 'Blue']
#     car_models = ['Prius', 'Camry', 'Corolla', 'RAV4']

#     def submit_form():
#         form_data = {
#             'Year': [year_var.get()],
#             'Car Makes': [make_var.get()],
#             'Tax Type': [tax_var.get()],
#             'Condition': [condition_var.get()],
#             'Body Type': [body_var.get()],
#             'Fuel': [fuel_var.get()],
#             'Transmission': [transmission_var.get()],
#             'Color': [color_var.get()],
#             'Car Model': [model_var.get()]
#         }
#         messagebox.showinfo("Form Data", str(form_data))

#     # Create form fields
#     # Year
#     ttk.Label(main_frame, text="Year:").grid(row=0, column=0, sticky=tk.W, pady=5)
#     year_combo = ttk.Combobox(main_frame, textvariable=year_var, values=years)
#     year_combo.grid(row=0, column=1, sticky=(tk.W, tk.E), pady=5)

#     # Car Makes
#     ttk.Label(main_frame, text="Car Make:").grid(row=1, column=0, sticky=tk.W, pady=5)
#     make_combo = ttk.Combobox(main_frame, textvariable=make_var, values=car_makes)
#     make_combo.grid(row=1, column=1, sticky=(tk.W, tk.E), pady=5)

#     # Tax Type
#     ttk.Label(main_frame, text="Tax Type:").grid(row=2, column=0, sticky=tk.W, pady=5)
#     tax_combo = ttk.Combobox(main_frame, textvariable=tax_var, values=tax_types)
#     tax_combo.grid(row=2, column=1, sticky=(tk.W, tk.E), pady=5)

#     # Condition
#     ttk.Label(main_frame, text="Condition:").grid(row=3, column=0, sticky=tk.W, pady=5)
#     condition_combo = ttk.Combobox(main_frame, textvariable=condition_var, values=conditions)
#     condition_combo.grid(row=3, column=1, sticky=(tk.W, tk.E), pady=5)

#     # Body Type
#     ttk.Label(main_frame, text="Body Type:").grid(row=4, column=0, sticky=tk.W, pady=5)
#     body_combo = ttk.Combobox(main_frame, textvariable=body_var, values=body_types)
#     body_combo.grid(row=4, column=1, sticky=(tk.W, tk.E), pady=5)

#     # Fuel
#     ttk.Label(main_frame, text="Fuel:").grid(row=5, column=0, sticky=tk.W, pady=5)
#     fuel_combo = ttk.Combobox(main_frame, textvariable=fuel_var, values=fuels)
#     fuel_combo.grid(row=5, column=1, sticky=(tk.W, tk.E), pady=5)

#     # Transmission
#     ttk.Label(main_frame, text="Transmission:").grid(row=6, column=0, sticky=tk.W, pady=5)
#     transmission_combo = ttk.Combobox(main_frame, textvariable=transmission_var, values=transmissions)
#     transmission_combo.grid(row=6, column=1, sticky=(tk.W, tk.E), pady=5)

#     # Color
#     ttk.Label(main_frame, text="Color:").grid(row=7, column=0, sticky=tk.W, pady=5)
#     color_combo = ttk.Combobox(main_frame, textvariable=color_var, values=colors)
#     color_combo.grid(row=7, column=1, sticky=(tk.W, tk.E), pady=5)

#     # Car Model
#     ttk.Label(main_frame, text="Car Model:").grid(row=8, column=0, sticky=tk.W, pady=5)
#     model_combo = ttk.Combobox(main_frame, textvariable=model_var, values=car_models)
#     model_combo.grid(row=8, column=1, sticky=(tk.W, tk.E), pady=5)

#     # Submit Button
#     submit_btn = ttk.Button(main_frame, text="Submit", command=submit_form)
#     submit_btn.grid(row=9, column=0, columnspan=2, pady=20)

#     return root
# # Create the form and display it
# root = create_form()

# # This will allow the Tkinter window to run in the background without blocking the Jupyter notebook
# root.after(1, root.mainloop)
# display(root)

In [281]:
# import tkinter as tk
# from tkinter import ttk
# from tkinter import messagebox
# import tkinter.font as tkfont

# def create_form():
#     root = tk.Tk()
#     root.title("Car Details Form")
#     root.geometry("500x700")
#     root.configure(bg='#f0f0f0')
    
#     style = ttk.Style()
#     style.configure('Header.TLabel', font=('Helvetica', 16, 'bold'), padding=10)
#     style.configure('Custom.TFrame', background='#f0f0f0')
#     style.configure('Custom.TLabel', font=('Helvetica', 10), padding=5)
#     style.configure('Submit.TButton', font=('Helvetica', 11, 'bold'), padding=10)
    
#     main_frame = ttk.Frame(root, padding="20", style='Custom.TFrame')
#     main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
    
#     header_label = ttk.Label(main_frame, text="Car Registration Form", style='Header.TLabel')
#     header_label.grid(row=0, column=0, columnspan=2, pady=(0, 20))

#     # Variables for form fields
#     year_var = tk.StringVar(value="2010")
#     make_var = tk.StringVar(value="Toyota")
#     tax_var = tk.StringVar(value="Tax Paper")
#     condition_var = tk.StringVar(value="New")
#     body_var = tk.StringVar(value="SUV")
#     fuel_var = tk.StringVar(value="Petrol")
#     transmission_var = tk.StringVar(value="Auto")
#     color_var = tk.StringVar(value="White")
#     model_var = tk.StringVar(value="Prius")

#     # Lists for dropdowns
#     years = [str(year) for year in range(2000, 2026)]
#     car_makes = ['Toyota', 'Honda', 'Ford', 'BMW', 'Mercedes', 'Audi', 'Volkswagen', 'Hyundai', 'Kia', 'Mazda']
#     tax_types = ['Tax Paper', 'No Paper']
#     conditions = ['New', 'Used']
#     body_types = ['SUV', 'Sedan', 'Hatchback', 'Pickup', 'Van', 'Coupe', 'Convertible', 'Wagon']
#     fuels = ['Petrol', 'Diesel', 'Hybrid', 'Electric', 'CNG', 'LPG']
#     transmissions = ['Auto', 'Manual', 'CVT', 'DCT']
#     colors = ['White', 'Black', 'Silver', 'Red', 'Blue', 'Gray', 'Green', 'Brown', 'Yellow', 'Orange']
#     car_models = ['Prius', 'Camry', 'Corolla', 'RAV4', 'Civic', 'Accord', 'CR-V', 'Focus', 'Mustang', 'F-150']

#     def submit_form():
#         form_data = {
#             'Year': [year_var.get()],
#             'Car Makes': [make_var.get()],
#             'Tax Type': [tax_var.get()],
#             'Condition': [condition_var.get()],
#             'Body Type': [body_var.get()],
#             'Fuel': [fuel_var.get()],
#             'Transmission': [transmission_var.get()],
#             'Color': [color_var.get()],
#             'Car Model': [model_var.get()]
#         }
#         messagebox.showinfo("Form Data", str(form_data))

#     def get_matches(event, combobox, all_values):
#         current_text = event.widget.get().lower()
#         matches = [value for value in all_values if current_text in value.lower()]
#         combobox['values'] = matches
#         # Open the dropdown automatically when user types
#         combobox.event_generate('<Down>')

#     fields = [
#         ("Year", year_var, years),
#         ("Car Make", make_var, car_makes),
#         ("Tax Type", tax_var, tax_types),
#         ("Condition", condition_var, conditions),
#         ("Body Type", body_var, body_types),
#         ("Fuel", fuel_var, fuels),
#         ("Transmission", transmission_var, transmissions),
#         ("Color", color_var, colors),
#         ("Car Model", model_var, car_models)
#     ]

#     form_container = ttk.Frame(main_frame, style='Custom.TFrame')
#     form_container.grid(row=1, column=0, columnspan=2, sticky=(tk.W, tk.E))
    
#     # Add form fields with search functionality
#     for idx, (label_text, var, values) in enumerate(fields):
#         field_frame = ttk.Frame(form_container, style='Custom.TFrame')
#         field_frame.grid(row=idx, column=0, sticky=(tk.W, tk.E), pady=5)
        
#         label = ttk.Label(field_frame, text=f"{label_text}:", style='Custom.TLabel')
#         label.grid(row=0, column=0, sticky=tk.W, padx=(0, 10))
        
#         combo = ttk.Combobox(field_frame, textvariable=var, values=values, width=30)
#         combo.grid(row=0, column=1, sticky=(tk.W, tk.E))
        
#         # Bind the KeyRelease event to update suggestions
#         combo.bind('<KeyRelease>', lambda e, cb=combo, vals=values: get_matches(e, cb, vals))
        
#         # Add tooltip
#         tooltip = tk.Label(field_frame, 
#                          text="Type to search options",
#                          bg='lightyellow',
#                          relief='solid',
#                          borderwidth=1)
        
#         def show_tooltip(event, tip=tooltip):
#             tip.grid(row=1, column=1, pady=(2, 0))
            
#         def hide_tooltip(event, tip=tooltip):
#             tip.grid_remove()
            
#         combo.bind('<FocusIn>', show_tooltip)
#         combo.bind('<FocusOut>', hide_tooltip)

#     # Submit Button
#     submit_frame = ttk.Frame(main_frame, style='Custom.TFrame')
#     submit_frame.grid(row=2, column=0, columnspan=2, pady=20)
    
#     submit_btn = ttk.Button(
#         submit_frame, 
#         text="Submit Details", 
#         command=submit_form,
#         style='Submit.TButton'
#     )
#     submit_btn.grid(row=0, column=0, pady=10)

#     # Configure grid weights
#     root.grid_columnconfigure(0, weight=1)
#     root.grid_rowconfigure(0, weight=1)
#     main_frame.grid_columnconfigure(1, weight=1)

#     return root

# # Create the form and display it
# root = create_form()

# # Run the Tkinter event loop without blocking the Jupyter notebook
# root.after(100, root.mainloop)


In [282]:
# import tkinter
# tkinter._test()
